# CMS Medicare Geographic Variation — SQL Analysis
**Project:** P1 — CMS Claims & Health Equity Analysis  
**Notebook:** 02 — SQL Analysis via SQLite  
**Goal:** Translate the Python-based exploration into SQL queries to demonstrate fluency with both tools. Queries identify high-need counties, anomalous spending patterns, and ER intensity profiles.

---

## 1. Setup — Load Data & Create SQLite Database

In [ ]:
import pandas as pd
import sqlite3

# Reload the raw CMS dataset — same source as notebook 01
df = pd.read_csv(r'C:\Users\ibrah\Downloads\Medicare Geographic Variation - by National, State & County\Medicare Geographic Variation - by National, State & County\2023\2014-2023 Medicare Fee-for-Service Geographic Variation Public Use File.csv')

# Filter to county-level rows for 2023 (our cross-sectional analysis year)
county_2023 = df[(df['BENE_GEO_LVL'] == 'County') & (df['YEAR'] == 2023)]

print("Rows:", len(county_2023))
print("Ready")

In [ ]:
# Create an in-memory SQLite database to run SQL queries against the CMS data
# SQLite lets us write and document real SQL without needing a database server
conn = sqlite3.connect('cms_equity.db')

# Write the filtered dataframe into a table called cms_county
# if_exists='replace' drops and recreates the table on each run — safe for a local analysis db
county_2023.to_sql('cms_county', conn, if_exists='replace', index=False)

# Verify the table loaded correctly by checking row count
result = pd.read_sql('SELECT COUNT(*) as total_rows FROM cms_county', conn)
print(result)

## 2. Spot Check — Preview Key Columns

In [ ]:
# Quick preview of the core equity columns before any filtering
# Confirms column names are accessible as-is through SQL without type casting
result = pd.read_sql("""
    SELECT BENE_GEO_DESC, BENE_DUAL_PCT, ER_VISITS_PER_1000_BENES, TOT_MDCR_STDZD_PYMT_PC
    FROM cms_county
    LIMIT 10
""", conn)
print(result)

## 3. Highest Dual Eligibility Counties

In [ ]:
# Rank counties by dual eligibility rate (descending) to identify the most vulnerable populations
# Dual eligibility = enrolled in both Medicare and Medicaid, a strong proxy for poverty and disability
result = pd.read_sql("""
    SELECT 
        BENE_GEO_DESC, 
        BENE_DUAL_PCT, 
        ER_VISITS_PER_1000_BENES, 
        TOT_MDCR_STDZD_PYMT_PC,
        BENE_AVG_RISK_SCRE, 
        ACUTE_HOSP_READMSN_PCT
    FROM cms_county
    ORDER BY BENE_DUAL_PCT DESC
    LIMIT 20
""", conn)
print(result)

## 4. Composite Need Score — Top 20 Highest-Need Counties

This query builds a composite need score by combining two normalized equity indicators:
- **Dual eligibility rate** — captures poverty and low-income burden
- **Average risk score** — captures overall health complexity

Both are min-max normalized to a 0–1 scale so they contribute equally regardless of their original units. Counties scoring highest on both dimensions represent the greatest unmet need.

In [ ]:
# CTE 1 (min_max): Compute the min and max of each indicator across all counties
# Used to normalize values onto a 0-1 scale — prevents dual_pct from dominating due to scale differences
# CTE 2 (normalized): Cast columns to FLOAT (stored as text due to CMS suppression '*'),
#   filter out suppressed rows, and compute normalized versions of each indicator
# Final SELECT: sum the two normalized scores into a composite need_score and rank counties
result = pd.read_sql("""
    WITH min_max AS (
        SELECT 
            MIN(CAST(BENE_DUAL_PCT AS FLOAT))    AS dual_min,
            MAX(CAST(BENE_DUAL_PCT AS FLOAT))    AS dual_max,
            MIN(CAST(BENE_AVG_RISK_SCRE AS FLOAT)) AS risk_min,
            MAX(CAST(BENE_AVG_RISK_SCRE AS FLOAT)) AS risk_max
        FROM cms_county
        WHERE BENE_DUAL_PCT != '*'
          AND BENE_AVG_RISK_SCRE != '*'
    ),
    normalized AS (
        SELECT
            c.BENE_GEO_DESC                                          AS geo_descr, 
            CAST(c.BENE_DUAL_PCT AS FLOAT)                           AS dual_pct, 
            CAST(c.ER_VISITS_PER_1000_BENES AS FLOAT)                AS er_intensity, 
            CAST(c.TOT_MDCR_STDZD_PYMT_PC AS FLOAT)                  AS total_spending,
            CAST(c.BENE_AVG_RISK_SCRE AS FLOAT)                      AS risk_score, 
            CAST(c.ACUTE_HOSP_READMSN_PCT AS FLOAT)                  AS readmission_rate,
            (CAST(c.BENE_DUAL_PCT AS FLOAT) - m.dual_min) 
                / (m.dual_max - m.dual_min)                          AS dual_normalized,
            (CAST(c.BENE_AVG_RISK_SCRE AS FLOAT) - m.risk_min) 
                / (m.risk_max - m.risk_min)                          AS risk_normalized
        FROM cms_county c, min_max m
        WHERE c.BENE_DUAL_PCT != '*'
          AND c.BENE_AVG_RISK_SCRE != '*'
    )
    SELECT
        geo_descr,
        dual_pct,
        er_intensity,
        total_spending,
        risk_score,
        readmission_rate,
        dual_normalized,
        risk_normalized,
        (dual_normalized + risk_normalized) AS need_score
    FROM normalized
    ORDER BY need_score DESC
    LIMIT 20
""", conn)
print(result)

## 5. High Spending, Low Dual Eligibility Counties

These counties spend above the national average but serve a below-average share of low-income beneficiaries. High spending here is less likely to reflect poverty-driven need and more likely to reflect supply-side factors (specialist density, high-cost procedures, regional pricing).

> **Finding:** High Medicare spending is not a reliable indicator of high need. The *type* of spending matters more than the amount.

In [ ]:
# Subqueries compute the national average for spending and dual eligibility
# Counties meeting both conditions represent high-cost, lower-vulnerability areas
# Useful for contrasting against high-need counties in the equity narrative
result = pd.read_sql("""
    SELECT 
        BENE_GEO_DESC, 
        BENE_DUAL_PCT, 
        ER_VISITS_PER_1000_BENES, 
        TOT_MDCR_STDZD_PYMT_PC,
        BENE_AVG_RISK_SCRE, 
        ACUTE_HOSP_READMSN_PCT
    FROM cms_county
    WHERE TOT_MDCR_STDZD_PYMT_PC > (
            SELECT AVG(CAST(TOT_MDCR_STDZD_PYMT_PC AS FLOAT)) FROM cms_county
          )
      AND BENE_DUAL_PCT < (
            SELECT AVG(CAST(BENE_DUAL_PCT AS FLOAT)) FROM cms_county
          )
    ORDER BY TOT_MDCR_STDZD_PYMT_PC DESC
    LIMIT 20
""", conn)
print(result)

## 6. National Averages — Baseline Reference

In [ ]:
# National average readmission rate — used as the threshold in filtering queries below
# CMS benchmark for acute readmission is ~15-16%; deviations suggest access or quality issues
result = pd.read_sql("""
    SELECT AVG(CAST(ACUTE_HOSP_READMSN_PCT AS FLOAT)) AS avg_readmission_rate
    FROM cms_county
""", conn)
print(result)

In [ ]:
# National average total standardized Medicare spending per capita
# Standardized payments adjust for geographic price differences — comparable across counties
result = pd.read_sql("""
    SELECT AVG(CAST(TOT_MDCR_STDZD_PYMT_PC AS FLOAT)) AS avg_total_spending
    FROM cms_county
""", conn)
print(result)

In [ ]:
# National average readmission rate (duplicate check — consolidate or remove before final push)
result = pd.read_sql("""
    SELECT AVG(CAST(ACUTE_HOSP_READMSN_PCT AS FLOAT)) AS avg_readmission_rate
    FROM cms_county
""", conn)
print(result)

## 7. High Spending, High Dual Eligibility Counties

Counties that are both high-cost *and* high-need. These represent the most resource-intensive equity challenge — high vulnerability paired with high utilization and spending. Compare this profile against Section 5 to distinguish need-driven from supply-driven spending.

In [ ]:
# Mirror of Section 5 — same spending threshold, but dual eligibility is above average
# Counties appearing here have both elevated poverty burden and elevated Medicare costs
result = pd.read_sql("""
    SELECT 
        BENE_GEO_DESC, 
        BENE_DUAL_PCT, 
        ER_VISITS_PER_1000_BENES, 
        TOT_MDCR_STDZD_PYMT_PC,
        BENE_AVG_RISK_SCRE, 
        ACUTE_HOSP_READMSN_PCT
    FROM cms_county
    WHERE TOT_MDCR_STDZD_PYMT_PC > (
            SELECT AVG(CAST(TOT_MDCR_STDZD_PYMT_PC AS FLOAT)) FROM cms_county
          )
      AND BENE_DUAL_PCT > (
            SELECT AVG(CAST(BENE_DUAL_PCT AS FLOAT)) FROM cms_county
          )
    ORDER BY TOT_MDCR_STDZD_PYMT_PC DESC
    LIMIT 20
""", conn)
print(result)

## 8. Highest ER Intensity Counties

In [ ]:
# Rank counties by ER visits per 1,000 beneficiaries
# High ER intensity often signals inadequate primary care access — a key health equity indicator
# CAST required because the column is stored as text due to CMS suppression encoding
results = pd.read_sql("""
    SELECT 
        BENE_GEO_DESC, 
        BENE_DUAL_PCT, 
        ER_VISITS_PER_1000_BENES, 
        TOT_MDCR_STDZD_PYMT_PC,
        BENE_AVG_RISK_SCRE, 
        ACUTE_HOSP_READMSN_PCT
    FROM cms_county
    ORDER BY CAST(ER_VISITS_PER_1000_BENES AS FLOAT) DESC
    LIMIT 20
""", conn)
print(results)

## 9. Counties with Above-Average Total Spending

In [ ]:
# Pull all counties exceeding the national average total standardized spending
# Used as a broad filter to isolate high-cost counties for further profiling
results = pd.read_sql("""
    SELECT 
        BENE_GEO_DESC,
        BENE_DUAL_PCT,
        CAST(ER_VISITS_PER_1000_BENES AS FLOAT)  AS er_visits_per_1000,
        CAST(TOT_MDCR_STDZD_PYMT_PC AS FLOAT)    AS total_spending,
        BENE_AVG_RISK_SCRE,
        ACUTE_HOSP_READMSN_PCT
    FROM cms_county
    WHERE CAST(TOT_MDCR_STDZD_PYMT_PC AS FLOAT) > (
        SELECT AVG(CAST(TOT_MDCR_STDZD_PYMT_PC AS FLOAT)) FROM cms_county
    )
""", conn)
print(results.shape)
print(results.head())

## 10. Above-Average Readmission Rate Counties

In [ ]:
# Pull counties where readmission rate exceeds the national average
# High readmissions alongside high risk scores may signal post-discharge support gaps
# Note: ACUTE_HOSP_READMSN_PCT comparison works here because values are already numeric
# after the type conversion in notebook 01 — but suppressed '*' rows will be excluded by SQLite's CAST
results = pd.read_sql("""
    SELECT 
        ACUTE_HOSP_READMSN_PCT, 
        BENE_AVG_RISK_SCRE
    FROM cms_county
    WHERE CAST(ACUTE_HOSP_READMSN_PCT AS FLOAT) > (
        SELECT AVG(CAST(ACUTE_HOSP_READMSN_PCT AS FLOAT)) FROM cms_county
    )
""", conn)
print(results)

## 11. Export Key Queries to `.sql` File

Saving the three core analytical queries as a standalone `.sql` file for portfolio documentation. This file can be linked from the README and demonstrates SQL proficiency independently of the notebook.

In [ ]:
sql_queries = """
-- Query 1: Composite Need Score — Top 20 Highest-Need Counties
-- Normalizes dual eligibility and average risk score to a 0-1 scale,
-- then sums them into a composite need score to rank counties by vulnerability.
WITH min_max AS (
    SELECT 
        MIN(CAST(BENE_DUAL_PCT AS FLOAT))      AS dual_min,
        MAX(CAST(BENE_DUAL_PCT AS FLOAT))      AS dual_max,
        MIN(CAST(BENE_AVG_RISK_SCRE AS FLOAT)) AS risk_min,
        MAX(CAST(BENE_AVG_RISK_SCRE AS FLOAT)) AS risk_max
    FROM cms_county
    WHERE BENE_DUAL_PCT != '*'
      AND BENE_AVG_RISK_SCRE != '*'
),
normalized AS (
    SELECT
        c.BENE_GEO_DESC,
        CAST(c.BENE_DUAL_PCT AS FLOAT)                          AS dual_pct,
        CAST(c.BENE_AVG_RISK_SCRE AS FLOAT)                     AS risk_score,
        CAST(c.ER_VISITS_PER_1000_BENES AS FLOAT)               AS er_visits,
        CAST(c.TOT_MDCR_STDZD_PYMT_PC AS FLOAT)                 AS total_spending,
        CAST(c.ACUTE_HOSP_READMSN_PCT AS FLOAT)                 AS readmission_pct,
        (CAST(c.BENE_DUAL_PCT AS FLOAT) - m.dual_min) 
            / (m.dual_max - m.dual_min)                         AS dual_normalized,
        (CAST(c.BENE_AVG_RISK_SCRE AS FLOAT) - m.risk_min) 
            / (m.risk_max - m.risk_min)                         AS risk_normalized
    FROM cms_county c, min_max m
    WHERE c.BENE_DUAL_PCT != '*'
      AND c.BENE_AVG_RISK_SCRE != '*'
)
SELECT
    BENE_GEO_DESC,
    ROUND(dual_pct, 4)                            AS dual_eligibility,
    ROUND(risk_score, 2)                          AS risk_score,
    ROUND(er_visits, 0)                           AS er_visits_per_1000,
    ROUND(total_spending, 2)                      AS total_spending_pc,
    ROUND(readmission_pct, 4)                     AS readmission_rate,
    ROUND(dual_normalized + risk_normalized, 4)   AS need_score
FROM normalized
ORDER BY need_score DESC
LIMIT 20;


-- Query 2: High Spending, Low Dual Eligibility Counties
-- Identifies counties with above-average spending but below-average poverty burden.
-- High costs here likely reflect supply-side factors rather than patient need.
SELECT 
    BENE_GEO_DESC, 
    BENE_DUAL_PCT, 
    ER_VISITS_PER_1000_BENES, 
    TOT_MDCR_STDZD_PYMT_PC,
    BENE_AVG_RISK_SCRE, 
    ACUTE_HOSP_READMSN_PCT
FROM cms_county
WHERE TOT_MDCR_STDZD_PYMT_PC > (
        SELECT AVG(CAST(TOT_MDCR_STDZD_PYMT_PC AS FLOAT)) FROM cms_county
      )
  AND BENE_DUAL_PCT < (
        SELECT AVG(CAST(BENE_DUAL_PCT AS FLOAT)) FROM cms_county
      )
ORDER BY TOT_MDCR_STDZD_PYMT_PC DESC
LIMIT 20;


-- Query 3: High ER Intensity County Profiles
-- Ranks counties by ER visit rate. High ER utilization relative to health burden
-- is a signal of inadequate primary and preventive care access.
SELECT 
    BENE_GEO_DESC,
    BENE_DUAL_PCT,
    CAST(ER_VISITS_PER_1000_BENES AS FLOAT) AS er_visits_per_1000,
    TOT_MDCR_STDZD_PYMT_PC,
    BENE_AVG_RISK_SCRE,
    ACUTE_HOSP_READMSN_PCT
FROM cms_county
WHERE CAST(ER_VISITS_PER_1000_BENES AS FLOAT) > (
    SELECT AVG(CAST(ER_VISITS_PER_1000_BENES AS FLOAT)) FROM cms_county
)
ORDER BY CAST(ER_VISITS_PER_1000_BENES AS FLOAT) DESC
LIMIT 20;
"""

with open(r'C:\Users\ibrah\Downloads\equity_analysis_queries.sql', 'w') as f:
    f.write(sql_queries)

print("SQL file saved successfully")

## 12. Pandas Need Score (Deferred)

The block below replicates the SQL composite need score in pandas and saves the processed file. Currently commented out — uncomment to regenerate `cms_county_processed.csv` if the working_df from notebook 01 is available in scope.

In [ ]:
# Replicate the composite need score calculation in pandas
# Useful for adding need_score as a column to working_df for downstream modeling or visualization

# dual_min = working_df['BENE_DUAL_PCT'].min()
# dual_max = working_df['BENE_DUAL_PCT'].max()
# risk_min = working_df['BENE_AVG_RISK_SCRE'].min()
# risk_max = working_df['BENE_AVG_RISK_SCRE'].max()

# working_df['dual_normalized'] = (working_df['BENE_DUAL_PCT'] - dual_min) / (dual_max - dual_min)
# working_df['risk_normalized'] = (working_df['BENE_AVG_RISK_SCRE'] - risk_min) / (risk_max - risk_min)
# working_df['need_score'] = (working_df['dual_normalized'] + working_df['risk_normalized']).round(4)

# Save processed file with need score added
# working_df.to_csv(r'C:\Users\ibrah\Downloads\cms_county_processed.csv', index=False)

# print("Processed file saved successfully")
# print("Shape:", working_df.shape)
# print("Columns:", working_df.columns.tolist())